# Local Inference — UNet Water Body Experiments

Runs inference from one or more trained experiment checkpoints on a local
PlanetScope scene. Outputs georeferenced GeoTIFF predictions alongside a
visual comparison across experiments.

**Expected directory layout:**
```
experiments/
  baseline_indices_only/
      best_model.pt
      norm_stats.npy
  optionB_bands_only/
      best_model.pt
      norm_stats.npy
  optionB_bands_and_indices/
      best_model.pt
      norm_stats.npy
```

**Memory note:** inference is run one chip at a time so RAM usage is
proportional to `CHIP_SIZE`, not the full scene size.

---
## 0. Configuration
Edit this cell only.

In [ ]:
from pathlib import Path

# ── Input ─────────────────────────────────────────────────────────────────────
# Path to the local 10-band PlanetScope GeoTIFF to run inference on
INPUT_IMAGE   = Path(r"C:\path\to\your\scene.tif")

# Root of the copied experiments folder
EXPERIMENTS_DIR = Path(r"C:\path\to\experiments")

# Output directory for predictions (created if it doesn't exist)
OUTPUT_DIR    = INPUT_IMAGE.parent / "predictions"

# ── Experiments to run ────────────────────────────────────────────────────────
# Each entry: (experiment_name, spectral_bands, display_label)
# spectral_bands must match what the model was trained on
EXPERIMENTS = [
    ("baseline_indices_only",      [6, 7, 8, 9],       "1: Indices only"),
    ("optionB_bands_only",         [0, 1, 2, 3, 4, 5], "2: Bands only"),
    ("optionB_bands_and_indices",  list(range(10)),     "3: Bands + Indices"),
]

# ── Model architecture (must match training config) ───────────────────────────
ENCODER_CHANNELS = [16, 32, 64, 128]
CHIP_SIZE        = 256

# ── Output ────────────────────────────────────────────────────────────────────
SAVE_GEOTIFF  = True    # save georeferenced prediction GeoTIFFs
SAVE_PLOT     = True    # save comparison figure as PNG

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Input : {INPUT_IMAGE}")
print(f"Output: {OUTPUT_DIR}")

---
## 1. Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import Patch
import warnings
warnings.filterwarnings("ignore")

import rasterio
from rasterio.windows import Window

import torch
import torch.nn as nn
import torch.nn.functional as F

DEVICE = torch.device("cpu")
print(f"Device: {DEVICE}")
print(f"PyTorch: {torch.__version__}")

---
## 2. Model Definition
Must match the architecture used during training.

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch,  out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.block(x)


class UNet(nn.Module):
    def __init__(self, in_channels, out_channels=2, encoder_chn=None):
        super().__init__()
        if encoder_chn is None:
            encoder_chn = [16, 32, 64, 128]
        self.encoders = nn.ModuleList()
        self.pools    = nn.ModuleList()
        prev = in_channels
        for ch in encoder_chn:
            self.encoders.append(DoubleConv(prev, ch))
            self.pools.append(nn.MaxPool2d(2))
            prev = ch
        self.bottleneck = DoubleConv(prev, prev * 2)
        prev = prev * 2
        self.upconvs  = nn.ModuleList()
        self.decoders = nn.ModuleList()
        for ch in reversed(encoder_chn):
            self.upconvs.append(nn.ConvTranspose2d(prev, ch, 2, stride=2))
            self.decoders.append(DoubleConv(ch * 2, ch))
            prev = ch
        self.output_conv = nn.Conv2d(prev, out_channels, 1)

    def forward(self, x):
        skips = []
        for enc, pool in zip(self.encoders, self.pools):
            x = enc(x); skips.append(x); x = pool(x)
        x = self.bottleneck(x)
        for up, dec, skip in zip(self.upconvs, self.decoders, reversed(skips)):
            x = up(x)
            if x.shape != skip.shape:
                x = F.pad(x, [0, skip.shape[3]-x.shape[3], 0, skip.shape[2]-x.shape[2]])
            x = dec(torch.cat([skip, x], dim=1))
        return self.output_conv(x)


print("UNet defined.")

---
## 3. Inference Utilities

In [ ]:
def load_norm_stats(stats_path: Path):
    """
    Load normalisation stats from .npy file.
    Handles three formats:
      - {"mean": array, "std": array}         old flat format
      - {"mukherjee": (mean, std), ...}       per-source format
      - {"global": (mean, std)}               global keyed format
    Always returns (mean, std) arrays for the local source.
    """
    stats = np.load(stats_path, allow_pickle=True).item()
    if "mean" in stats:
        return stats["mean"], stats["std"]
    if "local" in stats:
        return stats["local"]
    if "mukherjee" in stats:
        # No local stats saved yet — fall back to mukherjee stats with a warning
        print("  Warning: no 'local' stats found — using 'mukherjee' stats.")
        print("  For best results, compute local stats from your own scenes.")
        return stats["mukherjee"]
    # Fall back to whatever key exists
    key = next(iter(stats))
    print(f"  Warning: using stats key '{key}'")
    return stats[key]


def get_chip_offsets(total: int, chip_size: int):
    """Offsets covering full extent including partial final chip."""
    offsets = list(range(0, total - chip_size + 1, chip_size))
    if not offsets or offsets[-1] + chip_size < total:
        offsets.append(total - chip_size)
    return offsets


def load_chip(src, row: int, col: int, chip_size: int, band_indices: list):
    """Read a chip from an open rasterio dataset."""
    window = Window(col, row, chip_size, chip_size)
    data   = src.read([b + 1 for b in band_indices], window=window)
    return data.astype(np.float32)


def predict_scene(img_path: Path, model, spectral_bands: list,
                  mean: np.ndarray, std: np.ndarray,
                  chip_size: int = 256, device=DEVICE):
    """
    Run full-scene inference by tiling, predicting, and stitching.
    No-data pixels (identified via NaN in the NDWI band, index 7) are
    masked to 255 in the output.

    Returns uint8 array: 0=not-water, 1=water, 255=no-data
    """
    model.eval()

    with rasterio.open(img_path) as src:
        H, W = src.height, src.width
        # No-data mask from NDWI band (band 8, 1-indexed) which has NaN no-data
        ndwi = src.read(8, masked=False).astype(np.float32)
        nodata_mask = np.isnan(ndwi)

    prob_map  = np.zeros((H, W), dtype=np.float32)
    count_map = np.zeros((H, W), dtype=np.uint8)

    with rasterio.open(img_path) as src:
        for r in get_chip_offsets(H, chip_size):
            for c in get_chip_offsets(W, chip_size):
                chip = load_chip(src, r, c, chip_size, spectral_bands)
                chip = (chip - mean[:, None, None]) / std[:, None, None]
                chip = np.nan_to_num(chip, nan=0.0, posinf=1.0, neginf=-1.0)
                tensor = torch.from_numpy(chip).unsqueeze(0).to(device)
                with torch.no_grad():
                    prob = torch.softmax(model(tensor), dim=1)[0, 1].cpu().numpy()
                prob_map[r:r+chip_size,  c:c+chip_size] += prob
                count_map[r:r+chip_size, c:c+chip_size] += 1

    pred = (prob_map / np.maximum(count_map, 1) > 0.5).astype(np.uint8)
    pred[nodata_mask] = 255
    return pred


def save_geotiff(pred: np.ndarray, reference_path: Path, output_path: Path):
    """Write prediction as a georeferenced GeoTIFF matching the input scene."""
    with rasterio.open(reference_path) as src:
        profile = src.profile.copy()
    profile.update(count=1, dtype="uint8", compress="lzw", nodata=255)
    with rasterio.open(output_path, "w", **profile) as dst:
        dst.write(pred[np.newaxis, ...])
    print(f"  Saved: {output_path.name}")


print("Inference utilities defined.")

---
## 4. Load Image Metadata & False Colour Preview

In [ ]:
with rasterio.open(INPUT_IMAGE) as src:
    print(f"File  : {INPUT_IMAGE.name}")
    print(f"Shape : {src.height} × {src.width} px  ({src.count} bands)")
    print(f"CRS   : {src.crs}")
    print(f"Res   : {src.res}")
    print(f"dtype : {src.dtypes[0]}")
    full_img = src.read(masked=False).astype(np.float32)

# False colour: NIR-R-G (bands 6,4,3 → 0-indexed 5,3,2)
fc      = full_img[[5, 3, 2]].transpose(1, 2, 0)
p2, p98 = np.nanpercentile(fc, (2, 98))
fc      = np.clip((fc - p2) / (p98 - p2 + 1e-6), 0, 1)

fig, ax = plt.subplots(figsize=(8, 8))
ax.imshow(fc)
ax.set_title(f"False Colour (NIR-R-G)\n{INPUT_IMAGE.name}", fontsize=11)
ax.axis("off")
plt.tight_layout()
plt.show()

---
## 5. Run Inference

In [ ]:
predictions = []   # list of (display_label, pred_array)

for exp_name, spectral_bands, display_label in EXPERIMENTS:
    ckpt_dir   = EXPERIMENTS_DIR / exp_name
    model_path = ckpt_dir / "best_model.pt"
    stats_path = ckpt_dir / "norm_stats.npy"

    print(f"\n── {display_label} ──────────────────────")

    # Validate files exist
    if not model_path.exists():
        print(f"  Skipping — best_model.pt not found at {model_path}")
        continue
    if not stats_path.exists():
        print(f"  Skipping — norm_stats.npy not found at {stats_path}")
        continue

    # Load normalisation stats
    mean, std = load_norm_stats(stats_path)
    print(f"  Bands : {spectral_bands}")
    print(f"  mean  : {np.round(mean, 3)}")
    print(f"  std   : {np.round(std,  3)}")

    # Load model
    model = UNet(in_channels=len(spectral_bands),
                 encoder_chn=ENCODER_CHANNELS).to(DEVICE)
    model.load_state_dict(
        torch.load(model_path, map_location=DEVICE, weights_only=True)
    )
    print(f"  Loaded: {model_path.name}")

    # Run inference
    pred = predict_scene(
        INPUT_IMAGE, model, spectral_bands, mean, std,
        chip_size=CHIP_SIZE, device=DEVICE
    )
    predictions.append((display_label, exp_name, pred))

    water_pct = (pred == 1).sum() / (pred != 255).sum() * 100
    print(f"  Water: {water_pct:.2f}% of valid pixels")

    # Save GeoTIFF
    if SAVE_GEOTIFF:
        out_name = f"{INPUT_IMAGE.stem}_{exp_name}_pred.tif"
        save_geotiff(pred, INPUT_IMAGE, OUTPUT_DIR / out_name)

print(f"\nInference complete — {len(predictions)} experiment(s) run.")

---
## 6. Visual Comparison

In [ ]:
n_exp     = len(predictions)
mask_cmap = mcolors.ListedColormap(["white", "steelblue", "lightgrey"])

# Layout: first column = false colour, remaining columns = one per experiment
n_cols = n_exp + 1
fig, axes = plt.subplots(1, n_cols, figsize=(5 * n_cols, 8))

# False colour
axes[0].imshow(fc)
axes[0].set_title("False Colour\n(NIR-R-G)", fontsize=10, fontweight="bold")
axes[0].axis("off")

# Predictions
for col_i, (display_label, exp_name, pred) in enumerate(predictions, start=1):
    pred_display = np.where(pred == 255, 2, pred)
    water_pct    = (pred == 1).sum() / (pred != 255).sum() * 100
    axes[col_i].imshow(pred_display, cmap=mask_cmap, vmin=0, vmax=2,
                       interpolation="nearest")
    axes[col_i].set_title(f"{display_label}\nwater={water_pct:.1f}%",
                           fontsize=10, fontweight="bold")
    axes[col_i].axis("off")

legend_elements = [
    Patch(facecolor="white",     edgecolor="grey", label="Not water"),
    Patch(facecolor="steelblue", edgecolor="grey", label="Water"),
    Patch(facecolor="lightgrey", edgecolor="grey", label="No data"),
]
fig.legend(handles=legend_elements, loc="lower center", ncol=3,
           fontsize=10, framealpha=0.9, bbox_to_anchor=(0.5, 0.01))

plt.suptitle(INPUT_IMAGE.stem, fontsize=13, y=1.01)
plt.tight_layout(rect=[0, 0.05, 1, 1])

if SAVE_PLOT:
    plot_path = OUTPUT_DIR / f"{INPUT_IMAGE.stem}_comparison.png"
    plt.savefig(plot_path, dpi=150, bbox_inches="tight")
    print(f"Plot saved: {plot_path}")

plt.show()

---
## 7. Per-Experiment Water Fraction Summary

In [ ]:
import pandas as pd

rows = []
for display_label, exp_name, pred in predictions:
    valid      = pred != 255
    water_px   = (pred == 1).sum()
    valid_px   = valid.sum()
    nodata_px  = (pred == 255).sum()
    rows.append({
        "experiment":    display_label,
        "water_px":      int(water_px),
        "valid_px":      int(valid_px),
        "nodata_px":     int(nodata_px),
        "water_pct":     round(water_px / valid_px * 100, 2) if valid_px > 0 else 0,
        "nodata_pct":    round(nodata_px / pred.size * 100, 2),
    })

df = pd.DataFrame(rows).set_index("experiment")
display(df)